In [56]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

<font size="6" color="red"> ch14. 웹 데이터 수집</font>
<!-- # <span style="color:red">ch14. 웹 데이터 수집</span> -->

# BeautifulSoup과 parser
    (정적 웹크롤링, 공공 api 사용)
    
`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함되어있음
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [2]:
import requests # http요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [7]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/data/ch14_sample.html
response

<Response [200]>

In [8]:
if response:
    print("해당 url에 접근함")
else:
    print("해당 url에 거부됨")

해당 url에 접근함


In [9]:
response.status_code
#200 : 정상
# 404 : 없는 페이지

200

In [10]:
response.content # html의 바이너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [11]:
print(response.content.decode("utf-8"))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [13]:
print(response.text)

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [14]:
#html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, #response.content,
                    "html.parser")
soup

<!DOCTYPE html>

<html lang="en">
<head>
<meta charset="utf-8"/>
</head>
<body>
<h1 class="greeting css" id="text">Hello, CSS</h1>
<h1 class="css">Hi, CSS</h1>
<div id="subject">subject 선택자 안의 내용</div>
<p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
<div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
<div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>

In [32]:
# soup.select_one("선택자") : 해당 선택자 처음 하나 엘리먼트만
el = soup.select_one("h1.greeting.css")
print("el = ", el)
print("el.text =>", el.text)
print("el.string =>", el.string)
print("el의 속성들 =>", el.attrs)
print("el의 class속성 =>", el.attrs["class"])
print("el의 class속성 =>", el.attrs.get("class"))
#print("el의 href속성 없는 속성은 에러 =>", el.attrs["href"]) 
print("el의 href속성 =>", el.attrs.get("href"))
print("el의 name =>", el.name)

el =  <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class속성 => ['greeting', 'css']
el의 class속성 => ['greeting', 'css']
el의 href속성 => None
el의 name => h1


In [38]:
# 2. soup.select("선택자") : 해당 선택자 엘리먼트 다 list로
els = soup.select("h1.css")
print("els=>", els)
print("els들의 text", [el.text for el in els])
print("els들의 string", [el.string for el in els])
print("els들의 속성들", [el.attrs for el in els])
print("els들의 class 속성들", [el.attrs.get("clas") for el in els])

els=> [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text ['Hello, CSS', 'Hi, CSS']
els들의 string ['Hello, CSS', 'Hi, CSS']
els들의 속성들 [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성들 [None, None]


In [41]:
# 3. soup.find(태그, 속성) vs soup.select_one("선택자") : 해당 갖은 태그 처음 하나만
print("select_one : ", soup.select_one("h1.css"))
print("find       : ", soup.find("h1", {"class":"css"}))
print("find       : ", soup.find("h1", class_="css"))
print()
print("select_one : ", soup.select_one("h1#text"))
print("select_one : ", soup.find("h1", {"id":"text"}))

select_one :  <h1 class="greeting css" id="text">Hello, CSS</h1>
find       :  <h1 class="greeting css" id="text">Hello, CSS</h1>
find       :  <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one :  <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one :  <h1 class="greeting css" id="text">Hello, CSS</h1>


In [47]:
# 4. soup.find_all(태그, 속성) vs soup.select("선택자") : 해당 엘리먼트 다 list로
print("모든 h1.css와 span태그 : ", soup.select("h1.css, span"))
#print("모든 h1.css.greeting와 span태그 : ", soup.select("h1.css.greeting, span"))
print("모든 h1.css와 span태그 : ", soup.find_all(["h1"], class_="css") +
                                    soup.find_all("span"))

모든 h1.css와 span태그 :  [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css와 span태그 :  [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [51]:
# 없는 엘리먼트 찾기
print("find_all(빈list) : ", soup.find_all("a"))
print("find(None)       : ", soup.find("a"))
print("select(빈list) : ", soup.select("a"))
print("select_one(None) : ", soup.select_one("a"))

find_all(빈list) :  []
find(None)       :  None
select(빈list) :  []
select_one(None) :  None


# 정적 웹 데이터 수집(정적 웹크롤링)
## BeautifulSoup 모듈을 활용한html 웹 데이터 수집
### 환율정보 가져오기(네이버 증권 > 시장지표)

-https://finance.naver.com/marketindex/

    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더
        - Disallow : 크롤링 제한 폴더

In [60]:
# soup객체 생성 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://finance.naver.com/marketindex/'
response = requests.get(url)
# response # Response 
print(response.status_code)
# response.text # response.content
soup = BeautifulSoup(response.text, 'html.parser')

200


In [59]:
# soup객체 생성 방법2
from urllib.request import urlopen
url = 'https://finance.naver.com/marketindex/'
response = urlopen(url)
# response # HTTPResponse
print(response.status)
# print(response.read().decode('cp949'))
soup = BeautifulSoup(response, 'html.parser')

200


In [69]:
p = "1,417,000.70"
float(p.replace(",", "")) # 방법 1
float("".join(p.split(","))) # 방법 2

1417000.7

In [73]:
# div.head_info 밑의 span.value (find계열)
prices = []
headinfos = soup.find_all("div", class_="head_info")
for headinfo in headinfos:
#     print(headinfo)
#     print("-----------")
    price = headinfo.find("span", class_="value")
    prices.append(float("".join(price.text.split(","))))
print(prices)

[1416.1, 888.14, 1633.61, 209.86, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.92, 4441.1, 200543.23]


In [77]:
# span.value (find계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

[1416.1, 888.1, 1633.6, 209.9, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 200543.2]


In [78]:
# 금액들 : div.head_info 밑의 span.value 
price_els = soup.select('div.head_info > span.value')
len(price_els)

12

In [79]:
# 타이틀
title_els = soup.select('h3.h_lst > span.blind')
len(title_els)

12

In [80]:
# 단위들 : div.head_info > span > span.blind
unit_els = soup.select('div.head_info > span > span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [82]:
# 상승/하락 :  div.head_info > span.blind
trend_els = soup.select("div.head_info > span.blind")
#trend_els

In [83]:
len(title_els), len(price_els), len(units), len(trend_els)

(12, 12, 12, 12)

In [84]:
for idx in range(len(title_els)):
    print("{} : {} {} - {}".format(title_els[idx].text,
                                  price_els[idx].text,
                                  units[idx],
                                  trend_els[idx].text))

미국 USD : 1,416.10 원 - 상승
일본 JPY(100엔) : 888.14 원 - 상승
유럽연합 EUR : 1,633.61 원 - 상승
중국 CNY : 209.86 원 - 상승
달러/일본 엔 : 159.2400 엔 - 상승
유로/달러 : 1.1541 달러 - 하락
영국 파운드/달러 : 1.3507 달러 - 하락
달러인덱스 : 99.7100  - 상승
WTI : 83.2 달러 - 상승
휘발유 : 1863.92 원 - 하락
국제 금 : 4441.1 달러 - 상승
국내 금 : 200543.23 원 - 상승


In [87]:
# 위에 방법 말고 이 방법을 사용
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    print("{} : {}{} - {}".format(title.text, price.text, unit, trend.text))

미국 USD : 1,416.10원 - 상승
일본 JPY(100엔) : 888.14원 - 상승
유럽연합 EUR : 1,633.61원 - 상승
중국 CNY : 209.86원 - 상승
달러/일본 엔 : 159.2400엔 - 상승
유로/달러 : 1.1541달러 - 하락
영국 파운드/달러 : 1.3507달러 - 하락
달러인덱스 : 99.7100 - 상승
WTI : 83.2달러 - 상승
휘발유 : 1863.92원 - 하락
국제 금 : 4441.1달러 - 상승
국내 금 : 200543.23원 - 상승


In [91]:
import pandas as pd;
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append({"title" : title.text,
                 "price" : float(price.text.replace(",","")),
                 "unit" : unit,
                 "trend" : trend.text})
pd.DataFrame(data)#.to_csv("data/file.csv", index=False) <=파일로 따로 저장

,title,price,unit,trend
0,미국 USD,1416.1000,원,상승
1,일본 JPY(100엔),888.1400,원,상승
2,유럽연합 EUR,1633.6100,원,상승
3,중국 CNY,209.8600,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


In [94]:
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append([title.text, float(price.text.replace(",","")), unit, trend.text])
pd.DataFrame(data, columns=["title","price","unit","trend"])

,title,price,unit,trend
0,미국 USD,1416.1000,원,상승
1,일본 JPY(100엔),888.1400,원,상승
2,유럽연합 EUR,1633.6100,원,상승
3,중국 CNY,209.8600,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


### 이번 주 로또번호 출력
- 방법2에서 User Agent를 추가하여 soup생성
- https://search.daum.net/search?w=tot&DA=UME&t__nil_searchbox=btn&sugo=13&sq=lott&o=3&q=lotto (다음에서 lotto검색)
```
    1236회(2026.08.08 추첨)
    당첨번호 [12, 18, 21, 29, 34, 38]
    보너스번호 10 
```

In [97]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = "https://search.daum.net/search?w=tot&DA=UME&t__nil_searchbox=btn&sugo=13&sq=lott&o=3&q=lotto"
response = requests.get(url)
print("response의 상태 :", response.status_code)
soup = BeautifulSoup(response.text, "html.parser")
#soup

response의 상태 : 200


In [102]:
# 방법2
from urllib.request import urlopen, Request
headers = {"User-Agent":
           "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"}
request = Request(url, headers=headers)

response = urlopen(request)
print("response의 상태 :", response.status)
soup = BeautifulSoup(response, "html.parser")
#soup

response의 상태 : 200


In [122]:
# 1236회(2026.08.08 추첨)
#당첨번호 [12, 18, 21, 29, 34, 38]
#보너스 10
times = soup.select_one("div.prize span.f_red").text
date = soup.select_one("div.prize > span.date").text
title1 = soup.select_one("div.prize > strong").text[-4:]
lottonum = soup.select("div.lottonum > span.ball:nth-last-child(n+3)")
lottonum = soup.select("div.lottonum > span.ball")[:-2]
title2 = soup.select_one("div.lottonum span.screen_out").text
bonus_number = soup.select_one("div.lottonum > span.bg_ball1").text
print(times, date)
print(title1, [int(numbers.text) for numbers in lottonum])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [129]:
# 위의 select계열 함수를 find계열함수로 변경하여 구현해보기
# times = soup.select_one("div.prize span.f_red").text
prize = soup.find("div", class_="prize")
times = prize.find("span", class_="f_red").text

#date = soup.select_one("div.prize > span.date").text
date = prize.find("span", class_="date").text

#title1 = soup.select_one("div.prize > strong").text[-4:]
title1 = prize.find("strong").text[-4:]

#lotto_numbers = soup.select("div.lottonum > span.ball")[:-2]
lottonum = soup.find("div", class_="lottonum")
lotto_numbers = lottonum.find_all("span", class_="ball")[:-2]

#title2 = soup.select_one("div.lottonum span.screen_out").text
title2 = lottonum.find("span", class_="screen_out").text

#bonus_number = soup.select_one("div.lottonum > span.bg_ball1").text
bonus_number = lottonum.find("span", class_="bg_ball1").text

print(times, date)
print(title1,[int(numbers.text)for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


### 다음 뉴스 검색 리스트
```
no title   href
0  타이틀1  http://~
1  타이틀2  http://~
2  타이틀3  http://~
```

In [132]:
# 방법1
import requests
from bs4 import BeautifulSoup
word = "디워"
url = f"https://search.daum.net/search?w=news&q={word}"
print(url)
response = requests.get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, "html.parser")

https://search.daum.net/search?w=news&q=디워
200


In [136]:
# 방법2
from urllib.request import urlopen, Request
word = "디워"
url = f"https://search.daum.net/search?w=news&q={word}"
headers = {"User-Agent":
           "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36"}
#request = Request(url, headers=headers)
request = Request(url)
request.add_header("User-Agent",
                   "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36")
response = urlopen(request)
print(response.status)

UnicodeEncodeError: 'ascii' codec can't encode characters in position 21-22: ordinal not in range(128)

In [147]:
items_find_list = [] # 검색한 결과를 담을 dict 리스트
items_el = soup.select("div.item-title > strong.tit-g > a")
for idx, item in enumerate(items_el):
    # print(idx, item.attrs['href'])
    items_find_list.append({"no":idx+1,
                           "title":item.text,
                           "link":item.attrs.get("href")})
import pandas as pd
pd.DataFrame(items_find_list).head()

,no,title,link
0,1,"""망작"" 혹평의 심형래 '디워' 역주행 열풍…넷플릭스 글로벌 6위",http://v.daum.net/v/20260812114737800
1,2,왜 19년 만에?…‘망작’ 혹평받았던 ‘디 워’ 넷플릭스서 폭풍 역주행,http://v.daum.net/v/20260812163441643
2,3,19년 만에 넷플릭스 덮친 이무기…심형래 ‘디워’ 글로벌 6위 역주행,http://v.daum.net/v/20260812142053660
3,4,"사라진 신화 ‘디 워’, 19년 만에 넷플릭스 역주행 왜?",http://v.daum.net/v/20260812154307053
4,5,"'충무로 혹평→빚더미' 심형래, 19년 만에 '활짝' 웃었다…'디 워' 글로벌 6...",http://v.daum.net/v/20260812105107487


In [149]:
items_find_list = [] # 검색한 결과를 담을 2차원 리스트
items_el = soup.select("div.item-title > strong.tit-g > a")
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get("href")])
pd.DataFrame(items_find_list, columns=["순번","기사제목","링크"])

,순번,기사제목,링크
0,0,"""망작"" 혹평의 심형래 '디워' 역주행 열풍…넷플릭스 글로벌 6위",http://v.daum.net/v/20260812114737800
1,1,왜 19년 만에?…‘망작’ 혹평받았던 ‘디 워’ 넷플릭스서 폭풍 역주행,http://v.daum.net/v/20260812163441643
2,2,19년 만에 넷플릭스 덮친 이무기…심형래 ‘디워’ 글로벌 6위 역주행,http://v.daum.net/v/20260812142053660
3,3,"사라진 신화 ‘디 워’, 19년 만에 넷플릭스 역주행 왜?",http://v.daum.net/v/20260812154307053
4,4,"'충무로 혹평→빚더미' 심형래, 19년 만에 '활짝' 웃었다…'디 워' 글로벌 6...",http://v.daum.net/v/20260812105107487
5,5,"'호프' 극찬한 죄?..팬에 욕먹은 여배우 ""커뮤니티 너무 보지 마세요"" 일침",http://v.daum.net/v/20260812163129495
6,6,혹평받던 ‘디 워’가 어쩌다…19년 만에 넷플릭스 글로벌 6위,http://v.daum.net/v/20260812144804214
7,7,"19년 만의 역주행 심형래 <디워>, 넥스트는 알아봤다",http://v.daum.net/v/20260811163249667
8,8,"심형래 '디 워', 개봉 19년 만에 넷플릭스서 '역주행'…글로벌 6위까지",http://v.daum.net/v/20260812152812272
9,9,"""진짜 이무기가 날았다""...심형래 '디워', 개봉 19년 만에 넷플릭스 세계 7...",http://v.daum.net/v/20260811105245488


In [150]:
# 다음 뉴스 검색함수(원하는 키워드, 원하는 페이지 수로)


def collect_list(keyword, page):
    pass

In [151]:
collect_list("ey컨설팅", 10)